# ⚙️ End-to-End ML Pipeline with Experiment Tracking
**Author:** Tharun Kumar Reddy Byreddy

**Goal:** Build a production-style automated ML pipeline comparing multiple models with full experiment tracking using MLflow.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 2. Generate and Explore Data

In [ ]:
np.random.seed(42)
n = 5000

df = pd.DataFrame({
    'age':              np.random.randint(18, 75, n),
    'income':           np.random.normal(55000, 20000, n).astype(int),
    'credit_score':     np.random.randint(300, 850, n),
    'loan_amount':      np.random.normal(15000, 8000, n).astype(int),
    'employment_years': np.random.randint(0, 40, n),
    'num_accounts':     np.random.randint(1, 10, n),
    'missed_payments':  np.random.randint(0, 10, n),
    'education':        np.random.choice(['High School','Bachelor','Master','PhD'], n),
    'loan_purpose':     np.random.choice(['Home','Car','Education','Personal'], n),
    'marital_status':   np.random.choice(['Single','Married','Divorced'], n),
})

default_prob = np.clip(
    0.3 - 0.002*(df['credit_score']-300)/550
    + 0.001*df['missed_payments']
    - 0.0001*df['income']/55000
    + 0.002*df['loan_amount']/15000,
    0.05, 0.95
)
df['default'] = np.random.binomial(1, default_prob)

print('Shape:', df.shape)
print('Default rate:', df['default'].mean().round(4))
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].hist(df['age'], bins=20, color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Distribution')

axes[0,1].hist(df['credit_score'], bins=20, color='coral', edgecolor='white')
axes[0,1].set_title('Credit Score Distribution')

axes[0,2].hist(df['income'], bins=20, color='green', edgecolor='white')
axes[0,2].set_title('Income Distribution')

axes[1,0].hist(df['loan_amount'], bins=20, color='purple', edgecolor='white')
axes[1,0].set_title('Loan Amount Distribution')

df['default'].value_counts().plot(kind='pie', ax=axes[1,1],
    labels=['No Default','Default'], autopct='%1.1f%%',
    colors=['steelblue','coral'])
axes[1,1].set_title('Target Distribution')

axes[1,2].hist(df['missed_payments'], bins=10, color='orange', edgecolor='white')
axes[1,2].set_title('Missed Payments Distribution')

plt.tight_layout()
plt.savefig('../results/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing & Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

df_processed = df.copy()

# Feature engineering
df_processed['debt_to_income']     = df['loan_amount'] / (df['income'] + 1)
df_processed['payment_history']    = df['missed_payments'] / (df['num_accounts'] + 1)
df_processed['credit_utilization'] = df['loan_amount'] / (df['credit_score'] + 1)
df_processed['age_income_ratio']   = df['age'] / (df['income'] + 1)

# Encode categoricals
cat_cols = ['education', 'loan_purpose', 'marital_status']
le = LabelEncoder()
for col in cat_cols:
    df_processed[col] = le.fit_transform(df_processed[col])

# Scale numerics
num_cols = ['age','income','credit_score','loan_amount',
            'employment_years','num_accounts','missed_payments',
            'debt_to_income','payment_history',
            'credit_utilization','age_income_ratio']
scaler = StandardScaler()
df_processed[num_cols] = scaler.fit_transform(df_processed[num_cols])

X = df_processed.drop('default', axis=1)
y = df_processed['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Features:', X.shape)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 5. Train and Compare Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(random_state=42, eval_metric='logloss'),
    'SVM':                 SVC(probability=True, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    results[name] = {
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
        'F1 Score':  round(f1_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
    }
    print(f'{name:25s} | AUC: {results[name]["ROC-AUC"]} | F1: {results[name]["F1 Score"]}')

results_df = pd.DataFrame(results).T
print('\n=== Model Comparison ===')
results_df.sort_values('ROC-AUC', ascending=False)

## 6. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
colors = ['steelblue','coral','green','purple']

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison')
plt.legend()
plt.tight_layout()
plt.savefig('../results/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance

In [ ]:
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:12]

plt.figure(figsize=(10, 5))
plt.bar(range(len(indices)), importances[indices],
        color='steelblue', edgecolor='white')
plt.xticks(range(len(indices)),
           [X.columns[i] for i in indices],
           rotation=45, ha='right')
plt.title('Top Feature Importances — Random Forest')
plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Results Summary

In [ ]:
best = results_df['ROC-AUC'].idxmax()
print('='*50)
print('       ML PIPELINE RESULTS SUMMARY')
print('='*50)
print(results_df.sort_values('ROC-AUC', ascending=False).to_string())
print('='*50)
print(f'Best Model  : {best}')
print(f'Best AUC    : {results_df["ROC-AUC"].max():.4f}')
print('='*50)
print('Author: Tharun Kumar Reddy Byreddy')
print('M.S. Statistical Data Science | SFSU')